## Taller: Procesamiento de Datos en Infraestructura Cloud
## Unidad 3 – Evidencia de Aprendizaje EA3

**Dataset:** Netflix Movies and TV Shows  
**Fuente:** [Kaggle – https://www.kaggle.com/datasets/shivamb/netflix-shows

**Estudiante:** Juan Camilo Valencia Gutiérrez

**Docente:** Aharon Alexander Aguas 

**Fecha:** 2026

--

## Tabla de Contenido

| # | Sección |
|---|--------|
| 0 | Diseño del Esquema de Datos |
| 1 | Configuración y Evidencia de Databricks CE |
| 2 | Ingesta desde Kaggle y Creación de Tabla |
| 3 | Validaciones con Spark y SQL |
| 4 | Comparación SQL vs Spark |

---
# Sección 0 – Diseño del Esquema de Datos

## Descripción del Dataset

El dataset **Netflix Movies and TV Shows** contiene información sobre el catálogo de Netflix: películas y series disponibles, con detalles como país de origen, fecha de incorporación, director, reparto, duración y clasificación.

## Diccionario de Datos

| Campo | Tipo Spark | Tipo SQL | Nullable | Descripción |
|-------|-----------|----------|----------|-------------|
| `show_id` | StringType | STRING | No | Identificador único del título (clave primaria) |
| `type` | StringType | STRING | No | Tipo de contenido: `Movie` o `TV Show` |
| `title` | StringType | STRING | No | Nombre del título |
| `director` | StringType | STRING | Sí | Nombre del director (puede ser nulo) |
| `cast` | StringType | STRING | Sí | Lista de actores principales |
| `country` | StringType | STRING | Sí | País de producción |
| `date_added` | StringType | STRING | Sí | Fecha en que se agregó a Netflix |
| `release_year` | IntegerType | INT | No | Año de estreno original |
| `rating` | StringType | STRING | Sí | Clasificación por edades (PG-13, TV-MA, etc.) |
| `duration` | StringType | STRING | Sí | Duración: minutos (película) o temporadas (serie) |
| `listed_in` | StringType | STRING | Sí | Géneros/categorías del título |
| `description` | StringType | STRING | Sí | Sinopsis breve del contenido |

## Diagrama Entidad-Relación (simplificado)

```
┌──────────────────────────────────────────┐
│              netflix_titles              │
├──────────────┬───────────┬───────────────┤
│ show_id (PK) │ STRING    │ NOT NULL      │
│ type         │ STRING    │ NOT NULL      │
│ title        │ STRING    │ NOT NULL      │
│ director     │ STRING    │ NULLABLE      │
│ cast         │ STRING    │ NULLABLE      │
│ country      │ STRING    │ NULLABLE      │
│ date_added   │ STRING    │ NULLABLE      │
│ release_year │ INTEGER   │ NOT NULL      │
│ rating       │ STRING    │ NULLABLE      │
│ duration     │ STRING    │ NULLABLE      │
│ listed_in    │ STRING    │ NULLABLE      │
│ description  │ STRING    │ NULLABLE      │
└──────────────┴───────────┴───────────────┘
```

In [0]:
import subprocess
result = subprocess.run(
    ['find', '/Workspace', '-name', 'netflix_titles.csv'], 
    capture_output=True, text=True
)
print(result.stdout)

/Workspace/Users/juan.valenciag@est.iudigital.edu.co/netflix_titles.csv



In [0]:
# SECCIÓN 0 – Definición del esquema con StructType (PySpark)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

netflix_schema = StructType([
    StructField("show_id",      StringType(),  nullable=False),
    StructField("type",         StringType(),  nullable=False),
    StructField("title",        StringType(),  nullable=False),
    StructField("director",     StringType(),  nullable=True),
    StructField("cast",         StringType(),  nullable=True),
    StructField("country",      StringType(),  nullable=True),
    StructField("date_added",   StringType(),  nullable=True),
    StructField("release_year", IntegerType(), nullable=False),
    StructField("rating",       StringType(),  nullable=True),
    StructField("duration",     StringType(),  nullable=True),
    StructField("listed_in",    StringType(),  nullable=True),
    StructField("description",  StringType(),  nullable=True)
])

print("Esquema PySpark (StructType) definido correctamente:")
for field in netflix_schema.fields:
    nullable_str = "NULLABLE" if field.nullable else "NOT NULL"
    print(f"  - {field.name:<15} {str(field.dataType):<15} {nullable_str}")

Esquema PySpark (StructType) definido correctamente:
  - show_id         StringType()    NOT NULL
  - type            StringType()    NOT NULL
  - title           StringType()    NOT NULL
  - director        StringType()    NULLABLE
  - cast            StringType()    NULLABLE
  - country         StringType()    NULLABLE
  - date_added      StringType()    NULLABLE
  - release_year    IntegerType()   NOT NULL
  - rating          StringType()    NULLABLE
  - duration        StringType()    NULLABLE
  - listed_in       StringType()    NULLABLE
  - description     StringType()    NULLABLE


In [0]:
# SECCIÓN 0 – DDL equivalente en Spark SQL
ddl = """
CREATE TABLE IF NOT EXISTS netflix_titles (
    show_id      STRING  NOT NULL COMMENT 'Identificador unico del titulo',
    type         STRING  NOT NULL COMMENT 'Movie o TV Show',
    title        STRING  NOT NULL COMMENT 'Nombre del titulo',
    director     STRING           COMMENT 'Director del contenido',
    cast         STRING           COMMENT 'Actores principales',
    country      STRING           COMMENT 'Pais de produccion',
    date_added   STRING           COMMENT 'Fecha de ingreso a Netflix',
    release_year INT     NOT NULL COMMENT 'Anio de estreno',
    rating       STRING           COMMENT 'Clasificacion por edades',
    duration     STRING           COMMENT 'Duracion: min o temporadas',
    listed_in    STRING           COMMENT 'Generos del titulo',
    description  STRING           COMMENT 'Sinopsis breve'
)
USING DELTA
COMMENT 'Catalogo de contenido disponible en Netflix'
"""
print("DDL Spark SQL:")
print(ddl)

DDL Spark SQL:

CREATE TABLE IF NOT EXISTS netflix_titles (
    show_id      STRING  NOT NULL COMMENT 'Identificador unico del titulo',
    type         STRING  NOT NULL COMMENT 'Movie o TV Show',
    title        STRING  NOT NULL COMMENT 'Nombre del titulo',
    director     STRING           COMMENT 'Director del contenido',
    cast         STRING           COMMENT 'Actores principales',
    country      STRING           COMMENT 'Pais de produccion',
    date_added   STRING           COMMENT 'Fecha de ingreso a Netflix',
    release_year INT     NOT NULL COMMENT 'Anio de estreno',
    rating       STRING           COMMENT 'Clasificacion por edades',
    duration     STRING           COMMENT 'Duracion: min o temporadas',
    listed_in    STRING           COMMENT 'Generos del titulo',
    description  STRING           COMMENT 'Sinopsis breve'
)
USING DELTA
COMMENT 'Catalogo de contenido disponible en Netflix'



---
# Sección 1 – Configuración y Evidencia de Databricks CE

En esta sección se documenta el entorno de ejecución: versión del runtime, configuración del clúster, SparkContext y estructura de almacenamiento DBFS.


In [0]:
import sys
print(f"Spark version : {spark.version}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Compute type  : Serverless")

Spark version : 4.1.0
Python version: 3.11.10
Compute type  : Serverless


In [0]:
print("Configuraciones disponibles en Serverless:")
print(f"  shuffle.partitions : {spark.conf.get('spark.sql.shuffle.partitions', '200')}")
print(f"  Spark version      : {spark.version}")
print(f"  Compute type       : Serverless (sin acceso directo a SparkContext)")

Configuraciones disponibles en Serverless:
  shuffle.partitions : auto
  Spark version      : 4.1.0
  Compute type       : Serverless (sin acceso directo a SparkContext)


In [0]:
# ============================================================
# SECCIÓN 1 – Estructura de almacenamiento DBFS
# ============================================================
print("Estructura de almacenamiento en DBFS:")
print("-" * 50)

# Listamos el directorio raíz del DBFS
root_files = dbutils.fs.ls("/")
for f in root_files:
    print(f"  {f.path}")

print("\nContenido de /FileStore (carpeta de carga de archivos):")
print("-" * 50)
try:
    filestore = dbutils.fs.ls("/FileStore/")
    for f in filestore:
        print(f"  {f.path}")
except Exception as e:
    print(f"  (vacío o no existe aún) – {e}")

Estructura de almacenamiento en DBFS:
--------------------------------------------------
  dbfs:/Volumes/
  dbfs:/Workspace/
  dbfs:/databricks-datasets/

Contenido de /FileStore (carpeta de carga de archivos):
--------------------------------------------------
  (vacío o no existe aún) – [DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /FileStore SQLSTATE: 56038

JVM stacktrace:
com.databricks.backend.daemon.data.client.DbfsUnsupportedOperationSparkException
	at com.databricks.backend.daemon.data.client.DbfsExceptionMapperImpl.withExceptionWrapping(DbfsSparkException.scala:42)
	at com.databricks.backend.daemon.data.client.DBFSV2.listStatus(DatabricksFileSystemV2.scala:204)
	at com.databricks.backend.daemon.data.client.DatabricksFileSystem.listStatus(DatabricksFileSystem.scala:161)
	at com.databricks.sql.io.LokiFileSystem.$anonfun$listStatus$4(LokiFileSystem.scala:588)
	at com.databricks.sql.io.LokiFileSystem.tryWithNativeIO(LokiFileSystem.scala:622)
	at com.data

---
# Sección 2 – Ingesta desde Kaggle y Creación de Tabla

## Opciones de descarga del dataset

Se ofrecen **dos opciones** para obtener el dataset de Kaggle:

### Opción A – API de Kaggle (automatizada)
Requiere credenciales de Kaggle (`kaggle.json`) y la librería `kaggle` instalada.

### Opción B – Carga manual (recomendada para Databricks CE)
1. Descarga `netflix_titles.csv` desde https://www.kaggle.com/datasets/shivamb/netflix-shows
2. En Databricks: **Data → Add Data → Upload File → /FileStore/tables/**
3. El archivo queda disponible en `/Workspace/Users/juan.valenciag@est.iudigital.edu.co/netflix_titles.csv`

In [0]:
# SECCIÓN 2 – Carga del CSV en Spark con esquema definido
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

netflix_schema = StructType([
    StructField("show_id",      StringType(),  nullable=False),
    StructField("type",         StringType(),  nullable=False),
    StructField("title",        StringType(),  nullable=False),
    StructField("director",     StringType(),  nullable=True),
    StructField("cast",         StringType(),  nullable=True),
    StructField("country",      StringType(),  nullable=True),
    StructField("date_added",   StringType(),  nullable=True),
    StructField("release_year", IntegerType(), nullable=False),
    StructField("rating",       StringType(),  nullable=True),
    StructField("duration",     StringType(),  nullable=True),
    StructField("listed_in",    StringType(),  nullable=True),
    StructField("description",  StringType(),  nullable=True)
])

df = spark.read \
    .option("header", True) \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .schema(netflix_schema) \
.csv("/Workspace/Users/juan.valenciag@est.iudigital.edu.co/netflix_titles.csv")
print("    Dataset cargado exitosamente en Spark DataFrame")
print(f"   Total de registros : {df.count():,}")
print(f"   Total de columnas  : {len(df.columns)}")
print(f"   Particiones        : N/A (Serverless no expone RDD)")

    Dataset cargado exitosamente en Spark DataFrame
   Total de registros : 8,807
   Total de columnas  : 12
   Particiones        : N/A (Serverless no expone RDD)


In [0]:
# SECCIÓN 2 – Muestra de los primeros registros
print("Primeras 5 filas del DataFrame:")
df.show(5, truncate=50)

Primeras 5 filas del DataFrame:
+-------+-------+---------------------+---------------+--------------------------------------------------+-------------+------------------+------------+------+---------+--------------------------------------------------+--------------------------------------------------+
|show_id|   type|                title|       director|                                              cast|      country|        date_added|release_year|rating| duration|                                         listed_in|                                       description|
+-------+-------+---------------------+---------------+--------------------------------------------------+-------------+------------------+------------+------+---------+--------------------------------------------------+--------------------------------------------------+
|     s1|  Movie| Dick Johnson Is Dead|Kirsten Johnson|                                              NULL|United States|September 25, 2021|        2020|

In [0]:
# SECCIÓN 2 – Persistencia: crear tabla en Databricks (Delta)

# Primero eliminamos si ya existe (para poder re-ejecutar el notebook)
spark.sql("DROP TABLE IF EXISTS netflix_titles")

# Guardamos como tabla Delta en el metastore de Databricks
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("netflix_titles")

print("Tabla 'netflix_titles' creada y persistida en DBFS (formato Delta)")
print("\n Tablas disponibles en la base de datos:")
spark.sql("SHOW TABLES").show()

Tabla 'netflix_titles' creada y persistida en DBFS (formato Delta)

 Tablas disponibles en la base de datos:
+--------+--------------+-----------+
|database|     tableName|isTemporary|
+--------+--------------+-----------+
| default|netflix_titles|      false|
+--------+--------------+-----------+



---
# Sección 3 – Validaciones con Spark y SQL

En esta sección realizamos validaciones paralelas: la misma operación se ejecuta primero con **PySpark (DataFrame API)** y luego con **SQL**, comparando resultados. Esto permite verificar coherencia entre ambas interfaces y comprender las diferencias de sintaxis.

In [0]:
# SECCIÓN 3.1 – METADATOS: Schema con Spark
print("Schema del DataFrame (PySpark):")
df.printSchema()

Schema del DataFrame (PySpark):
root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [0]:
# SECCIÓN 3.1 – METADATOS: DESCRIBE TABLE con SQL
print("DESCRIBE TABLE (SQL):")
spark.sql("DESCRIBE TABLE netflix_titles").show(20, truncate=False)

DESCRIBE TABLE (SQL):
+------------+---------+-------+
|col_name    |data_type|comment|
+------------+---------+-------+
|show_id     |string   |NULL   |
|type        |string   |NULL   |
|title       |string   |NULL   |
|director    |string   |NULL   |
|cast        |string   |NULL   |
|country     |string   |NULL   |
|date_added  |string   |NULL   |
|release_year|int      |NULL   |
|rating      |string   |NULL   |
|duration    |string   |NULL   |
|listed_in   |string   |NULL   |
|description |string   |NULL   |
+------------+---------+-------+



In [0]:
# SECCIÓN 3.1 – METADATOS: SHOW CREATE TABLE
print("SHOW CREATE TABLE (SQL):")
spark.sql("SHOW CREATE TABLE netflix_titles").show(1, truncate=False)

SHOW CREATE TABLE (SQL):
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|createtab_stmt                                                                                                                              

In [0]:
# SECCIÓN 3.2 – DESCRIPCIÓN ESTADÍSTICA: df.describe() con Spark
print("Estadísticas descriptivas (PySpark df.describe()):")
df.describe(["release_year", "type", "rating", "country"]).show(truncate=False)

Estadísticas descriptivas (PySpark df.describe()):
+-------+------------------+-------+------+-----------------+
|summary|release_year      |type   |rating|country          |
+-------+------------------+-------+------+-----------------+
|count  |8807              |8807   |8803  |7976             |
|mean   |2014.1801975701146|NULL   |NULL  |NULL             |
|stddev |8.819312130833964 |NULL   |NULL  |NULL             |
|min    |1925              |Movie  |66 min|, France, Algeria|
|max    |2021              |TV Show|UR    |Zimbabwe         |
+-------+------------------+-------+------+-----------------+



In [0]:
# SECCIÓN 3.2 – DESCRIPCIÓN ESTADÍSTICA: Funciones agregadas SQL
print("Estadísticas con funciones SQL:")
spark.sql("""
    SELECT
        COUNT(*)                          AS total_registros,
        COUNT(DISTINCT type)              AS tipos_unicos,
        MIN(release_year)                 AS anio_min,
        MAX(release_year)                 AS anio_max,
        ROUND(AVG(release_year), 2)       AS anio_promedio,
        COUNT(CASE WHEN director IS NULL THEN 1 END) AS directores_nulos,
        COUNT(CASE WHEN country IS NULL THEN 1 END)  AS paises_nulos
    FROM netflix_titles
""").show(truncate=False)

Estadísticas con funciones SQL:
+---------------+------------+--------+--------+-------------+----------------+------------+
|total_registros|tipos_unicos|anio_min|anio_max|anio_promedio|directores_nulos|paises_nulos|
+---------------+------------+--------+--------+-------------+----------------+------------+
|8807           |2           |1925    |2021    |2014.18      |2634            |831         |
+---------------+------------+--------+--------+-------------+----------------+------------+



In [0]:
# SECCIÓN 3.3 – SELECT y FILTROS con PySpark
from pyspark.sql.functions import col

print("Películas de EE.UU. lanzadas después de 2015 (PySpark):")
df.select("title", "type", "country", "release_year", "rating") \
  .filter(
      (col("type") == "Movie") &
      (col("country") == "United States") &
      (col("release_year") > 2015)
  ) \
  .orderBy(col("release_year").desc()) \
  .show(10, truncate=40)

Películas de EE.UU. lanzadas después de 2015 (PySpark):
+----------------------------+-----+-------------+------------+------+
|                       title| type|      country|release_year|rating|
+----------------------------+-----+-------------+------------+------+
|                   Aftermath|Movie|United States|        2021| TV-MA|
|                  Sweet Girl|Movie|United States|        2021|     R|
|                The Starling|Movie|United States|        2021| PG-13|
|      Untold: Breaking Point|Movie|United States|        2021| TV-MA|
|Untold: Malice at the Palace|Movie|United States|        2021| TV-MA|
|                        Kate|Movie|United States|        2021|     R|
|                   Pray Away|Movie|United States|        2021| PG-13|
|                  Cosmic Sin|Movie|United States|        2021|     R|
|            The Paper Tigers|Movie|United States|        2021| PG-13|
|               The Water Man|Movie|United States|        2021|    PG|
+--------------------

In [0]:
# SECCIÓN 3.3 – SELECT y FILTROS con SQL (equivalente)
print("Películas de EE.UU. lanzadas después de 2015 (SQL):")
spark.sql("""
    SELECT title, type, country, release_year, rating
    FROM   netflix_titles
    WHERE  type = 'Movie'
      AND  country = 'United States'
      AND  release_year > 2015
    ORDER BY release_year DESC
    LIMIT 10
""").show(truncate=40)

Películas de EE.UU. lanzadas después de 2015 (SQL):
+----------------------------+-----+-------------+------------+------+
|                       title| type|      country|release_year|rating|
+----------------------------+-----+-------------+------------+------+
|                   Aftermath|Movie|United States|        2021| TV-MA|
|                  Sweet Girl|Movie|United States|        2021|     R|
|Untold: Malice at the Palace|Movie|United States|        2021| TV-MA|
|                   Pray Away|Movie|United States|        2021| PG-13|
|                        Kate|Movie|United States|        2021|     R|
|               The Water Man|Movie|United States|        2021|    PG|
|                The Starling|Movie|United States|        2021| PG-13|
|      Untold: Breaking Point|Movie|United States|        2021| TV-MA|
|            The Paper Tigers|Movie|United States|        2021| PG-13|
|                  Cosmic Sin|Movie|United States|        2021|     R|
+------------------------

In [0]:
# SECCIÓN 3.4 – GROUP BY: Conteo por tipo de contenido (PySpark)
from pyspark.sql.functions import count, round as spark_round

print("Distribución por tipo de contenido (PySpark):")
total = df.count()
df.groupBy("type") \
  .agg(count("*").alias("total")) \
  .withColumn("porcentaje", spark_round((col("total") / total) * 100, 2)) \
  .orderBy(col("total").desc()) \
  .show()

Distribución por tipo de contenido (PySpark):
+-------+-----+----------+
|   type|total|porcentaje|
+-------+-----+----------+
|  Movie| 6131|     69.62|
|TV Show| 2676|     30.38|
+-------+-----+----------+



In [0]:
# SECCIÓN 3.4 – GROUP BY: Conteo por tipo de contenido (SQL)
print("Distribución por tipo de contenido (SQL):")
spark.sql("""
    SELECT
        type,
        COUNT(*) AS total,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS porcentaje
    FROM netflix_titles
    GROUP BY type
    ORDER BY total DESC
""").show()

Distribución por tipo de contenido (SQL):
+-------+-----+----------+
|   type|total|porcentaje|
+-------+-----+----------+
|  Movie| 6131|     69.62|
|TV Show| 2676|     30.38|
+-------+-----+----------+



In [0]:
# SECCIÓN 3.4 – GROUP BY: Top 10 países productores (PySpark)
from pyspark.sql.functions import count

print("Top 10 países productores de contenido (PySpark):")
df.filter(col("country").isNotNull()) \
  .groupBy("country") \
  .agg(count("*").alias("titulos")) \
  .orderBy(col("titulos").desc()) \
  .limit(10) \
  .show(truncate=False)

Top 10 países productores de contenido (PySpark):
+--------------+-------+
|country       |titulos|
+--------------+-------+
|United States |2818   |
|India         |972    |
|United Kingdom|419    |
|Japan         |245    |
|South Korea   |199    |
|Canada        |181    |
|Spain         |145    |
|France        |124    |
|Mexico        |110    |
|Egypt         |106    |
+--------------+-------+



In [0]:
# SECCIÓN 3.4 – GROUP BY: Top 10 países productores (SQL)
print("Top 10 países productores de contenido (SQL):")
spark.sql("""
    SELECT country, COUNT(*) AS titulos
    FROM   netflix_titles
    WHERE  country IS NOT NULL
    GROUP BY country
    ORDER BY titulos DESC
    LIMIT 10
""").show(truncate=False)

Top 10 países productores de contenido (SQL):
+--------------+-------+
|country       |titulos|
+--------------+-------+
|United States |2818   |
|India         |972    |
|United Kingdom|419    |
|Japan         |245    |
|South Korea   |199    |
|Canada        |181    |
|Spain         |145    |
|France        |124    |
|Mexico        |110    |
|Egypt         |106    |
+--------------+-------+



In [0]:
# SECCIÓN 3.4 – GROUP BY: Producción por año (PySpark)
from pyspark.sql.functions import count

print("Cantidad de títulos por año de lanzamiento – últimos 10 años (PySpark):")
df.filter(col("release_year") >= 2014) \
  .groupBy("release_year") \
  .agg(count("*").alias("titulos")) \
  .orderBy(col("release_year").desc()) \
  .show(10)

Cantidad de títulos por año de lanzamiento – últimos 10 años (PySpark):
+------------+-------+
|release_year|titulos|
+------------+-------+
|        2021|    592|
|        2020|    953|
|        2019|   1030|
|        2018|   1147|
|        2017|   1032|
|        2016|    902|
|        2015|    560|
|        2014|    352|
+------------+-------+



In [0]:
# SECCIÓN 3.4 – GROUP BY: Producción por año (SQL)
print("Cantidad de títulos por año de lanzamiento – últimos 10 años (SQL):")
spark.sql("""
    SELECT release_year, COUNT(*) AS titulos
    FROM   netflix_titles
    WHERE  release_year >= 2014
    GROUP BY release_year
    ORDER BY release_year DESC
    LIMIT 10
""").show()

Cantidad de títulos por año de lanzamiento – últimos 10 años (SQL):
+------------+-------+
|release_year|titulos|
+------------+-------+
|        2021|    592|
|        2020|    953|
|        2019|   1030|
|        2018|   1147|
|        2017|   1032|
|        2016|    902|
|        2015|    560|
|        2014|    352|
+------------+-------+



In [0]:
# SECCIÓN 3.5 – Conteo de nulos por columna (PySpark)
from pyspark.sql.functions import col, sum as spark_sum, when, isnan

print("Conteo de valores nulos por columna:")
nulos = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])
nulos.show(vertical=True)

Conteo de valores nulos por columna:
-RECORD 0------------
 show_id      | 0    
 type         | 0    
 title        | 0    
 director     | 2634 
 cast         | 825  
 country      | 831  
 date_added   | 10   
 release_year | 0    
 rating       | 4    
 duration     | 3    
 listed_in    | 0    
 description  | 0    



---
# Sección 4 – Comparación: SQL vs Spark (PySpark)

Basado en la experiencia práctica de esta actividad, a continuación se presenta un análisis comparativo entre ambas interfaces para el procesamiento de datos.

## Tabla Comparativa

| Criterio | SQL (Spark SQL) | Spark (PySpark DataFrame API) |
|----------|----------------|-------------------------------|
| **Facilidad de uso** |  Alta: sintaxis declarativa familiar para analistas y DBAs. No requiere conocimientos de programación | Media: requiere conocimiento de Python y la API de DataFrame/RDD |
| **Expresividad** |  Muy natural para consultas ad hoc: JOINs, GROUP BY, Window Functions en pocas líneas |  Más flexible para lógica compleja: encadenamiento de transformaciones, condicionales anidados |
| **Escalabilidad y rendimiento** |  Ambos usan el mismo motor (Catalyst Optimizer), pero SQL puede generar planes subóptimos en consultas muy complejas |  Permite controlar particionado, caché y persistencia de forma más directa |
| **Pipelines y automatización** |  Limitado: difícil de parametrizar dinámicamente sin concatenación de strings |  Ideal para pipelines ETL: se integra nativamente con Python, loops, clases y librerías |
| **UDFs y ML** |  Soporta UDFs pero con sintaxis más verbosa (`CREATE FUNCTION`) |  Integración nativa con MLlib, Pandas UDFs, y el ecosistema Python (scikit-learn, etc.) |
| **Debugging y trazabilidad** |  Errores menos descriptivos; difícil de depurar consultas largas |  Mejor trazabilidad: se pueden inspeccionar DataFrames intermedios, usar `.explain()` |
| **Integración con BI** |  Compatible con herramientas como Power BI, Tableau, Metabase via JDBC/SQL |  Requiere transformar resultados a tabla o vista temporal para conexión con BI |

##  Análisis y Conclusión

Durante el desarrollo de esta práctica se evidenció que:

1. **Para consultas de exploración y validación** (DESCRIBE, COUNT, GROUP BY simples), **SQL es más rápido de escribir** y produce código más legible para alguien sin experiencia en programación.

2. **Para el procesamiento del pipeline** (carga del CSV, aplicación del esquema, detección de nulos, transformaciones condicionales), **PySpark fue más eficiente** porque permitió combinar lógica Python con operaciones distribuidas.

3. **En Databricks CE ambos enfoques son equivalentes en rendimiento** porque internamente SQL se traduce al mismo plan de ejecución Catalyst que la API de DataFrame. La diferencia es principalmente de **ergonomía y contexto de uso**.

4. **Recomendación práctica:** usar SQL para consultas de negocio y reportes, y PySpark para pipelines de ingeniería de datos, ML y operaciones que requieren lógica programática.

In [0]:
# SECCIÓN 4 – Demostración: misma lógica, diferente API
from pyspark.sql.functions import col, count, when

# --- PySpark ---
print("PySpark: Contenido por clasificación de rating (top 5):")
df.filter(col("rating").isNotNull()) \
  .groupBy("rating") \
  .agg(count("*").alias("titulos")) \
  .orderBy(col("titulos").desc()) \
  .limit(5) \
  .show()

# --- SQL ---
print("SQL equivalente: Contenido por clasificación de rating (top 5):")
spark.sql("""
    SELECT rating, COUNT(*) AS titulos
    FROM   netflix_titles
    WHERE  rating IS NOT NULL
    GROUP BY rating
    ORDER BY titulos DESC
    LIMIT 5
""").show()

print("\n Ambas consultas producen resultados idénticos.")
print("   Diferencia: SQL es más conciso; PySpark es más composable.")

PySpark: Contenido por clasificación de rating (top 5):
+------+-------+
|rating|titulos|
+------+-------+
| TV-MA|   3207|
| TV-14|   2160|
| TV-PG|    863|
|     R|    799|
| PG-13|    490|
+------+-------+

SQL equivalente: Contenido por clasificación de rating (top 5):
+------+-------+
|rating|titulos|
+------+-------+
| TV-MA|   3207|
| TV-14|   2160|
| TV-PG|    863|
|     R|    799|
| PG-13|    490|
+------+-------+


 Ambas consultas producen resultados idénticos.
   Diferencia: SQL es más conciso; PySpark es más composable.


---
## Conclusiones Finales

En esta actividad se logró:

- **Diseñar** un esquema de datos completo con `StructType` y DDL para el dataset de Netflix
- **Configurar** y documentar el entorno de Databricks Community Edition
- **Ingestar** el dataset desde Kaggle hacia DBFS y crear una tabla Delta persistente
- **Validar** los datos con múltiples consultas en PySpark y SQL
- **Comparar** las ventajas y desventajas de SQL vs Spark con evidencia práctica

El uso de Databricks CE como plataforma cloud permitió experimentar con un entorno real de Big Data, comprendiendo la arquitectura distribuida de Spark y la flexibilidad que ofrece trabajar tanto con la API programática de PySpark como con la interfaz declarativa de SQL.

---
*Notebook desarrollado para EA3 – Procesamiento de datos en infraestructura cloud*  
*Institución Universitaria Digital de Antioquia*